In [7]:
import numpy as np
import pandas as pd

# 📊 Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# 🔍 Statistical Libraries
from scipy.stats import norm, skew
from scipy import stats

# ⚠️ Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# 🎨 Set visualization style
sns.set(style="whitegrid")

from lightgbm import early_stopping, log_evaluation
import catboost as CAT
import lightgbm as LGBM
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold, train_test_split
# import ydf as tfdf # tensorflow decision forests
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from scipy.stats.mstats import winsorize
import pyinla
import pymc as pm
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Lasso
from sklearn.linear_model import RANSACRegressor, LinearRegression , LogisticRegression
from sklearn.datasets import make_regression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
# from ydata_profiling import ProfileReport
from sklearn.svm import SVR
from autoviz.AutoViz_Class import AutoViz_Class
from sklearn.ensemble import IsolationForest

%matplotlib inline
import dtale
import os 

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

models = []




In [8]:
train = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/train_features.csv')
test = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/test_features.csv')
train_labels = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/train_labels.csv')
train = pd.merge(train, train_labels, on='window_id', how='inner')
features = [feat for feat in test.columns if feat != 'window_id' ] # and feat != 'day']
train = train[list ( set (features + ['y']) - set(['pet','erc']) )]


In [ ]:
train.describe()

In [9]:
# # Data Analysis and fix
# 1. Identify Null values
# 2. Identify Skewed Data 
# 3. Identify outliers
# 4. Identify co-relation and drop the columsn which are highly co-related 
# 5. drop unique columsn or id columns as they add much value

# Identify Numerical columns (int, float)
numeric_cols = train.select_dtypes(include=['number']).columns.tolist()
# Identify Categorical columns (strings, objects, booleans)
categorical_cols = train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print ( "Columns with numberical values:{}".format ( numeric_cols))
print ( "Columns with Alphabetical values:{}".format ( categorical_cols))
print('NaN Count:', train[numeric_cols].isnull().sum().sum(), '\n')
print(train[categorical_cols].nunique(),'\n')


# Identify & Fix skewness 
skewed_feats = train.apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[skewed_feats > 0.75]
print ( "Skewed features identified with log transformation: ", skewed_feats.index.tolist() )
# Fix skewed features with log transformation
# ['pr', 'sph', 'vs', 'vpd']
# for feat in skewed_feats.index:
train['sph'] = np.log1p(train['sph'])


# Identify & Fix co-related columns
corr_matrix = train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]
print(f"Columns to consider dropping: {to_drop}")
                                  
# # Fix outliers 
np.random.seed(42)
def find_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

outlier , lower , high = find_outliers_iqr ( train , 'fm1000')



Columns with numberical values:['fm1000', 'srad', 'vpd', 'bi', 'tmmx', 'rmax', 'sph', 'day', 'vs', 'y', 'rmin', 'etr', 'fm100', 'tmmn', 'pr']
Columns with Alphabetical values:[]
NaN Count: 0 

Series([], dtype: float64) 

Skewed features identified with log transformation:  ['vpd', 'sph', 'vs', 'y', 'pr']
Columns to consider dropping: []


In [10]:
y               = train['y']
X               = train.drop(['y'], axis=1)
# test_feat          = ['sph', 'bi', 'fm100', 'etr', 'rmax', 'fm1000', 'rmin', 'vs', 'tmmn','tmmx', 'vpd', 'pr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42 , n_jobs=-1 )
rf.fit(X_train, y_train)
predictions = rf.predict(X_test)
print(f"MSE: {mean_squared_error(y_test, predictions)}")
print(f"R2 Score: {rf.score(X_test, y_test)}")

In [11]:
# We combine 3 different logical approaches to average out errors
reg1 = RandomForestRegressor(
    n_estimators=300, 
    max_depth=None,       # Allow full depth to capture complex logic
    random_state=42, 
    n_jobs=-1
)

reg2 = ExtraTreesRegressor(
    n_estimators=300, 
    max_depth=None, 
    random_state=42, 
    n_jobs=-1
)

reg3 = HistGradientBoostingRegressor(
    max_iter=500, 
    learning_rate=0.05, 
    max_depth=10, 
    random_state=42
)

# Voting Regressor averages the predictions of all three
ensemble = VotingRegressor([
    ('rf', reg1), 
    ('et', reg2), 
    ('hgb', reg3)
])

# 6. Train
ensemble.fit(X_train, y_train)

predictions = ensemble.predict(X_test)
print(f"MSE: {mean_squared_error(y_test, predictions)}")
print(f"R2 Score: {rf.score(X_test, y_test)}")

KeyboardInterrupt: 

In [ ]:
lr = LinearRegression(  n_jobs=-1 )
lr.fit(X_train, y_train)
predictions = lr.predict(X_test)
print(f"MSE: {mean_squared_error(y_test, predictions)}")
print(f"R2 Score: {lr.score(X_test, y_test)}")

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {'max_depth': [3, 5, 10, None], 'min_samples_split': [2, 5, 10]}
dt = DecisionTreeRegressor()
grid_search = GridSearchCV(dt, param_grid, cv=5, n_jobs=4)
grid_search.fit(X_train, y_train)
scores = cross_val_score(dt, X_test, y_test, cv=5, n_jobs=-1)

print ( scores.mean() )

In [ ]:
dt = DecisionTreeRegressor()
dt = DecisionTreeRegressor()
dt.fit(X_train, y_train)
predictions =  dt.predict(X_test)
print(f"MSE: {mean_squared_error(y_test, predictions)}")
print(f"R2 Score: {lr.score(X_test, y_test)}")

In [ ]:
import xgboost as xgb
# 3. Initialize and fit the XGBRegressor
model = xgb.XGBRegressor(
    objective='reg:squarederror', # Default for regression
    n_estimators=100,             # Number of boosting trees
    learning_rate=0.1,            # Step size shrinkage
    max_depth=5                   # Maximum tree depth
)
model.fit(X_train, y_train)

# 4. Make predictions and evaluate
predictions = model.predict(X_test)
rmse = root_mean_squared_error(y_test, predictions)
print(f"Root Mean Squared Error: {rmse}")

In [ ]:
train = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/train_features.csv')
test = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/test_features.csv')
train_labels = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/train_labels.csv')
train = pd.merge(train, train_labels, on='window_id', how='inner')

features = [feat for feat in train.columns if feat != 'window_id' and feat != 'day']
train = train[list ( set (features + ['y']) - set(['day']) )]
train =  train [['sph', 'bi', 'fm100', 'etr', 'rmax', 'fm1000', 'rmin', 'vs','tmmn', 'tmmx', 'vpd', 'pr','y']]
display ( train.shape )
train.drop_duplicates(inplace=True , ignore_index=True)

from autoviz import FixDQ
fixdq = FixDQ()
display ( train.shape )
display ( train.columns)
train = fixdq.fit_transform(train  )
test = fixdq.transform ( test )

#Impletment the outlier cap recommendation
# df_cleaned = fixdq.cap_outliers(train)

display ( train.shape)
display ( train.columns)

In [ ]:
train =  train [['sph', 'bi', 'fm100', 'etr', 'rmax', 'fm1000', 'rmin', 'vs', 'tmmn','tmmx', 'vpd', 'pr', 'y']]
features = [feat for feat in train.columns if feat != 'window_id' and feat != 'day']

y               = train['y']
X               = train.drop(['y'], axis=1)
test_feat          = ['sph', 'bi', 'fm100', 'etr', 'rmax', 'fm1000', 'rmin', 'vs', 'tmmn','tmmx', 'vpd', 'pr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [ ]:
display ( y_test.head())

In [ ]:

model = RandomForestRegressor ( n_estimators= 10)
model.fit( X , y )
test['y']  = model.predict (test [['sph', 'bi', 'fm100', 'etr', 'rmax', 'fm1000', 'rmin', 'vs', 'tmmn','tmmx', 'vpd', 'pr']])




In [ ]:
_id = 'window_id'
test            = test.drop_duplicates(subset=[_id])

In [ ]:
test

In [ ]:
if test.y.gt(1).any() | test.y.lt(0).any():
    print("Warning: Predicted values are out of bounds (0-1). Please check the predictions.")
    print ( [y for y in test['y'] if y > 1 or y < 0] )
    test['y'] = test['y'].clip(0, 1)
    test[[_id,'y']] . to_csv('submission.csv',index=False);
    !kaggle competitions submit spatiotemporal-wildfire-prediction -f /Users/vkdvamshi/gitRepos/ai-ml-learnings/submission.csv -m "submission"    
else:
    print("Predicted values are within the expected range (0-1). Ready for submission.")
    test[[_id,'y']] . to_csv('submission.csv',index=False);
    !kaggle competitions submit spatiotemporal-wildfire-prediction -f /Users/vkdvamshi/gitRepos/ai-ml-learnings/submission.csv -m "submission"

In [ ]:
import h2o 
from h2o.estimators import H2OGradientBoostingEstimator
from h2o.automl import H2OAutoML
h2o.init(max_mem_size="8G")
train = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/train_features.csv')
test = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/test_features.csv')
train_labels = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/train_labels.csv')
train = pd.merge(train, train_labels, on='window_id', how='inner')
from autoviz import FixDQ
fixdq = FixDQ()
display ( train.shape )
display ( train.columns)
train = fixdq.fit_transform(train  )

train.to_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/train.csv')

df = h2o.import_file ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/train.csv')
train, valid, test = df.split_frame(ratios=[0.7, 0.15], seed=1234)
predictors = ['day' , 'erc', 'srad', 'tmmx', 'etr', 'sph', 'vpd', 'vs', 'rmin', 'fm1000','pet', 'rmax', 'bi', 'pr', 'tmmn', 'fm100']
response = "y"

aml = H2OAutoML(max_models=100, max_runtime_secs=300, seed=1)
aml.train(x=predictors, y=response, training_frame=train)
lb = aml.leaderboard
print(lb.head(rows=lb.nrows))

# h2o.shutdown()

In [ ]:
# aml.predict ( test )

test['y'] = aml.predict ( test )



In [ ]:
h2o.export_file ( test[['window_id','y']] , path= '/Users/vkdvamshi/gitRepos/ai-ml-learnings/submissions.csv', force=True)

In [ ]:
!kaggle competitions submit spatiotemporal-wildfire-prediction -f /Users/vkdvamshi/gitRepos/ai-ml-learnings/submission.csv -m "submission"    

In [ ]:
class LassoRegression:
    def __init__(self, learning_rate=0.01, iterations=1000, l1_penalty=1.0):
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.l1_penalty = l1_penalty

    def fit(self, X, y):
        self.m, self.n = X.shape
        self.beta = np.zeros(self.n)
        self.bias = 0
        self.X = X
        self.y = y

        for _ in range(self.iterations):
            self.update_weights()

    def update_weights(self):
        y_pred = self.predict(self.X)
        d_beta = np.zeros(self.n)
        d_bias = -2 * np.sum(self.y - y_pred) / self.m

        for j in range(self.n):
            if self.beta[j] > 0:
                d_beta[j] = (-2 * np.dot(self.X[:, j], self.y - y_pred) + self.l1_penalty) / self.m
            else:
                d_beta[j] = (-2 * np.dot(self.X[:, j], self.y - y_pred) - self.l1_penalty) / self.m

        self.beta -= self.learning_rate * d_beta
        self.bias -= self.learning_rate * d_bias

    def predict(self, X):
        return np.dot(X, self.beta) + self.bias

In [ ]:
# 2. Define base models
base_models = [
    # ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    # ('svc', make_pipeline(StandardScaler(), SVC(probability=True))),
    ('svr', SVR()),
    # ('lg', Lasso(alpha=0.1))
]

# 3. Define Meta-Learner
meta_model = LogisticRegression()
# meta_model =  DecisionTreeRegressor(criterion='squared_error', max_depth=10, random_state=42)

# 4. Create Stacking Classifier
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    n_jobs=20 ,
    cv=4  # Cross-validation folds
)

# 5. Train and Evaluate
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
stacking_model.fit(X_train, y_train)

print(f"Accuracy: {stacking_model.score(X_test, y_test):.4f}")

In [ ]:
# 2. Basic Data Health Check
print("--- Dataset Info ---")
print(df.info())
print("\n--- Missing Values ---")
print(df.isnull().sum())
print("\n--- Descriptive Statistics ---")
print(df.describe().T)

# 3. Visualizing Distributions (Histograms)
# Focus on fire indices and meteorological drivers
cols_to_plot = ['tmmn', 'tmmx', 'erc', 'bi', 'vpd', 'vs']
df[cols_to_plot].hist(bins=20, figsize=(12, 8), color='skyblue', edgecolor='black')
plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.show()

# 4. Correlation Heatmap
# Crucial for ML to find redundant features (e.g., tmmn vs tmmx)
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()

# 5. Time-Series Trend (Daily change for a single window_id)
# Selecting the first window_id from your image as an example
sample_window = df['window_id'].unique()[0]
df_subset = df[df['window_id'] == sample_window]

plt.figure(figsize=(10, 5))
plt.plot(df_subset['day'], df_subset['erc'], marker='o', label='ERC (Energy Release)')
plt.plot(df_subset['day'], df_subset['bi'], marker='x', label='BI (Burning Index)')
plt.xlabel("Day")
plt.ylabel("Index Value")
plt.title(f"Fire Risk Trends for Window: {sample_window}")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

train = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/train_features.csv')
test = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/test_features.csv')
train_labels = pd.read_csv ( '/Users/vkdvamshi/gitRepos/ai-ml-learnings/wildFires -  ML/train_labels.csv')

train = pd.merge(train, train_labels, on='window_id', how='inner')
features = [feat for feat in test.columns if feat != 'window_id' and feat != 'day']
train = train[list ( set (features + ['y']) - set(['day']) )]

#Identify skewed features
skewed_feats = train[features].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[skewed_feats > 0.75]
print ( "Skewed features fixed with log transformation: ", skewed_feats.index.tolist() )
# Fix skewed features with log transformation
# ['pr', 'sph', 'vs', 'vpd']
# for feat in skewed_feats.index:
train['sph'] = np.log1p(train['sph'])

## Actual model fit and prediction 
y               = train['y']
X               = train[features]
test_feat          = test [features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# build Modles with different criteria and max_depth to see if it improves the score in kaggle
mse = {}
for model in models:
    print ( "Fitting model: ", model)
    mse[str(model)] = mean_squared_error ( y_test , model.fit(X,y).predict(X_test) )
    
min_key = min(mse, key=mse.get)
test['y']  = min_key.fit(X,y).predict(test_feat); 

display ( test.head() )

_id = 'window_id'
test            = test.drop_duplicates(subset=[_id])

if test.y.gt(1).any() | test.y.lt(0).any():
    print("Warning: Predicted values are out of bounds (0-1). Please check the predictions.")
    print ( [y for y in test['y'] if y > 1 or y < 0] )
    test['y'] = test['y'].clip(0, 1)
    test[[_id,'y']] . to_csv('submission.csv',index=False);
    !kaggle competitions submit spatiotemporal-wildfire-prediction -f /Users/vkdvamshi/gitRepos/ai-ml-learnings/submission.csv -m "submission"    
else:
    print("Predicted values are within the expected range (0-1). Ready for submission.")
    test[[_id,'y']] . to_csv('submission.csv',index=False);
    !kaggle competitions submit spatiotemporal-wildfire-prediction -f /Users/vkdvamshi/gitRepos/ai-ml-learnings/submission.csv -m "submission"

# Model to be experimented for classification
- Naive bayes
    - Alpha
    - Fit Prior
    - Binarize
- Logistic regression
    - L1/L2 Penality 
    - Class Weight
    - Solver
- k-nearest neighbor 
    - N Neighbors 
    - Weights 
    - Algorithm
- random forest
    - Criterion
    - Max Depth
    - Min Sample split
- support vector machines 
- Decision tree
    - Criterion
    - Max Depth 
    - Min Sample split

# Model to be experimented for Regression
- random forest 
- support vector machines 
- Decision tree 
    - Criterion
    - Max Depth 
    - Min Sample split
- Simple linear regression
    - L1/ L2 Penality
    - Fit intercept 
    - Solver  
- Logistic regression
    - L1/L2 Penality 
    - Class Weight
    - Solver
- Multi variante regression
- Lasso regression
- Gradient Boosted Trees
    - Criterion
    - Max Depth
    - N Estimators 
    - Max Features
-Principal Component
    - N component
    - Iterated Power
    - SVD Solver


# Xtreme Gradient Booster

In [ ]:
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', 
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5
)
# xgb_model.fit(X_train, y_train)
# xgb_val_preds = xgb_model.predict(X_test)
# xgb_mse = mean_squared_error(y_val, xgb_val_preds)
# print(f"XGBoost Validation MSE: {xgb_mse:.4f}")
# print(f"XGBoost Validation MAE: {mean_absolute_error(y_val, xgb_val_preds):.4f}")
# rmse = np.sqrt(mean_squared_error(y_val, val_preds))
# print(f"Validation RMSE: {rmse:.4f}")
# print(mean_absolute_error(y_val, val_preds))

print ( X_test)

# test['y'] = xgb_val_preds
# test[['window_id', 'y']].to_csv('submission.csv', index=False)
# import os 
# print ( os.getcwd() )

# Lasso regression 

In [ ]:
def score(y_pred, y_true):
    error = np.square(np.log10(y_pred +1) - np.log10(y_true +1)).mean() ** 0.5
    score = 1 - error
    return score

# actual_y = list(y_val['y'])
# actual_y = np.asarray(actual_y)


lasso_reg = Lasso(alpha=0.1, max_iter=10000, random_state=42)
lasso_reg.fit(X_train, y_train)
y_pred_lass =lasso_reg.predict(X_val)
# print("nnLasso SCORE : ", score(y_pred_lass, actual_y))
